In [93]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix

sys.path.append(os.path.abspath('ml-models'))
from mlmodels.Models import Models

from sklearn.metrics import confusion_matrix

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [94]:
#Defining random seeds
with open('./random_seeds.txt', 'r') as f:
    random_seeds = eval(f.read())

### Construct dataset

In [95]:
# Original dataset
data = pd.read_csv('data/tables/generated/features_dataset.csv', index_col=0)
# Use only static features
dynamic_features = data[[col for col in data.columns if 'min' in col or 'mean' in col]]
data = data.drop(columns=[col for col in data.columns if 'min' in col or 'mean' in col])
data

,segid,resid,mutation,phenotype,d_volume,d_hydropathy_KD,d_MW,d_pi,d_rogov,Rif_distance,...,phi,psi,n_hbond_acceptors,n_hbond_donors,SASA,snap2_score,deep_ddG,rasp_score,temp_factor,secondary_structure
2,C,639,E639D,0,-27.3,0.0,-14.0,-0.45,0.109,34.015193,...,-72.58,-54.68,0.0,0.0,107.910414,50,-0.152,0.254674,41.400002,0
3,C,113,V113I,0,26.7,0.3,14.1,0.06,0.494,24.626817,...,-55.57,-55.02,2.0,0.0,46.996241,-95,0.471,0.222572,36.919998,1
4,C,642,G642S,0,28.9,-0.4,30.0,-0.29,0.120,42.448172,...,169.90,-149.75,1.0,1.0,0.226439,48,-2.495,0.548394,32.490002,2
5,C,751,I751V,0,-26.7,-0.3,-14.1,-0.06,0.494,27.986912,...,-161.65,154.98,1.0,1.0,23.895392,-91,-0.242,0.287591,19.900000,2
6,C,944,K944N,0,-54.5,0.4,-14.1,-4.33,0.197,59.941562,...,60.77,28.12,0.0,0.0,208.845293,-73,0.025,0.215516,86.489998,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
262,C,445,H445F,1,36.7,6.0,10.0,-2.11,-0.016,9.683248,...,-66.73,-38.07,1.0,2.0,6.609119,92,-0.684,0.201490,16.459999,1
263,C,434,M434I,1,3.8,2.6,-18.0,0.28,-0.452,9.843524,...,-62.15,126.41,0.0,1.0,7.242104,93,-1.393,0.290369,17.299999,3
264,C,262,V262A,1,-51.4,-2.4,-28.0,0.04,0.232,56.595298,...,-126.69,153.98,0.0,0.0,95.032454,-82,-0.214,0.275822,133.350006,3
265,C,582,S582A,1,-0.4,2.6,-16.0,0.32,0.249,18.638826,...,-95.24,-170.26,1.0,0.0,0.000000,29,-1.548,0.237018,18.350000,0


### Select for mutations outside RRDR only and with four key features

In [96]:
#Mutations outside RRDR
outside_rrdr = data[((data.resid < 426) | (data.resid > 452))]
print(outside_rrdr)

    segid  resid mutation  phenotype  d_volume  d_hydropathy_KD  d_MW  d_pi  \
2       C    639    E639D          0     -27.3              0.0 -14.0 -0.45   
3       C    113    V113I          0      26.7              0.3  14.1  0.06   
4       C    642    G642S          0      28.9             -0.4  30.0 -0.29   
5       C    751    I751V          0     -26.7             -0.3 -14.1 -0.06   
6       C    944    K944N          0     -54.5              0.4 -14.1 -4.33   
..    ...    ...      ...        ...       ...              ...   ...   ...   
258     C    731    L731P          1     -54.0             -5.4 -16.1  0.32   
260     C    493    S493R          1      84.4             -3.7  69.1  5.08   
264     C    262    V262A          1     -51.4             -2.4 -28.0  0.04   
265     C    582    S582A          1      -0.4              2.6 -16.0  0.32   
266     C    378    L378R          1       6.7             -8.3  43.0  4.78   

     d_rogov  Rif_distance  ...     phi     psi  n_

In [97]:
feature_list = ['Rif_distance', 'RNA_distance', 'snap2_score', 'temp_factor']
features_and_label = feature_list + ['phenotype']

print ("Data with features:", features_and_label)

Data with features: ['Rif_distance', 'RNA_distance', 'snap2_score', 'temp_factor', 'phenotype']


In [98]:
data_outside_rrdr = outside_rrdr[features_and_label]
print(data_outside_rrdr)

     Rif_distance  RNA_distance  snap2_score  temp_factor  phenotype
2       34.015193     42.798608           50    41.400002          0
3       24.626817     35.774887          -95    36.919998          0
4       42.448172     49.686496           48    32.490002          0
5       27.986912     32.846026          -91    19.900000          0
6       59.941562     56.670554          -73    86.489998          0
..            ...           ...          ...          ...        ...
258     33.058292     23.530022           75    21.740000          1
260     13.468693     21.002388           26    20.940001          1
264     56.595298     65.031517          -82   133.350006          1
265     18.638826     23.573846           29    18.350000          1
266     17.636419     31.834717           60    21.850000          1

[216 rows x 5 columns]


In [99]:
phenotypes_nonRRDR = data_outside_rrdr["phenotype"]
phenotypes_nonRRDR

2      0
3      0
4      0
5      0
6      0
      ..
258    1
260    1
264    1
265    1
266    1
Name: phenotype, Length: 216, dtype: int64

In [100]:
x_nonRRDR = data_outside_rrdr[feature_list]
x_nonRRDR

,Rif_distance,RNA_distance,snap2_score,temp_factor
2,34.015193,42.798608,50,41.400002
3,24.626817,35.774887,-95,36.919998
4,42.448172,49.686496,48,32.490002
5,27.986912,32.846026,-91,19.900000
6,59.941562,56.670554,-73,86.489998
...,...,...,...,...
258,33.058292,23.530022,75,21.740000
260,13.468693,21.002388,26,20.940001
264,56.595298,65.031517,-82,133.350006
265,18.638826,23.573846,29,18.350000


### Get DT model trained on original dataset

In [101]:
model_type = ['DT']
all_models_results_original_DT = {model_type: [] for model_type in model_types}

for model_type in model_types:
    for seed in random_seeds:
        model = Models(
            {"all": data[features_and_label]},
            model_type,
            "recall",
            random_seed=seed,
            test_fraction=0.20,
            verbose=False,
            output_plots=False,
        )
        output_original_DT = model.returning_output(output_plots=False)
        all_models_results_original_DT[model_type].append(output_original_DT)

In [102]:
all_models_results_original_DT

{'DT': [{'Precision': 0.6666666666666666,
   'Sensitivity': 0.8888888888888888,
   'Specificity': 0.9090909090909091,
   'F1_Score': 0.761904761904762,
   'FPR': 0.09090909090909094,
   'VME': 9.090909090909092,
   'ME': 11.11111111111111,
   'Confusion_matrix': array([[40,  4],
          [ 1,  8]]),
   'ROC_AUC': 0.8952020202020202,
   'Precision_shifted': 0.6666666666666666,
   'Sensitivity_shifted': 0.8888888888888888,
   'Specificity_shifted': 0.9090909090909091,
   'F1_Score_shifted': 0.761904761904762,
   'FPR_shifted': 0.09090909090909094,
   'VME_shifted': 9.090909090909092,
   'ME_shifted': 11.11111111111111,
   'Confusion_matrix_shifted': array([[40,  4],
          [ 1,  8]])},
  {'Precision': 1.0,
   'Sensitivity': 0.8888888888888888,
   'Specificity': 1.0,
   'F1_Score': 0.9411764705882353,
   'FPR': 0.0,
   'VME': 0.0,
   'ME': 11.11111111111111,
   'Confusion_matrix': array([[44,  0],
          [ 1,  8]]),
   'ROC_AUC': 0.9406565656565656,
   'Precision_shifted': 1.0,
   

In [103]:
model_type = ['DT']
all_models_results_original_DT_models = {model_type: [] for model_type in model_types}

for model_type in model_types:
    for seed in random_seeds:
        model = Models(
            {"all": data[features_and_label]},
            model_type,
            "recall",
            random_seed=seed,
            test_fraction=0.20,
            verbose=False,
            output_plots=False,
        )
        output_original_DT_model = model.get_model()
        all_models_results_original_DT_models[model_type].append(output_original_DT_model)

In [104]:
all_models_results_original_DT_models

{'DT': [DecisionTreeClassifier(max_depth=2, random_state=0),
  DecisionTreeClassifier(max_depth=2, random_state=0),
  DecisionTreeClassifier(max_depth=2, random_state=0),
  DecisionTreeClassifier(max_depth=2, random_state=0),
  DecisionTreeClassifier(max_depth=2, random_state=0),
  DecisionTreeClassifier(max_depth=4, random_state=0),
  DecisionTreeClassifier(max_depth=2, random_state=0),
  DecisionTreeClassifier(max_depth=2, random_state=0),
  DecisionTreeClassifier(max_depth=2, random_state=0),
  DecisionTreeClassifier(max_depth=2, random_state=0),
  DecisionTreeClassifier(max_depth=2, random_state=0),
  DecisionTreeClassifier(max_depth=4, random_state=0),
  DecisionTreeClassifier(max_depth=2, random_state=0),
  DecisionTreeClassifier(max_depth=2, random_state=0),
  DecisionTreeClassifier(max_depth=2, random_state=0),
  DecisionTreeClassifier(max_depth=2, random_state=0),
  DecisionTreeClassifier(max_depth=2, random_state=0),
  DecisionTreeClassifier(max_depth=2, random_state=0),
  De

### Run DT models on non-RRDR dataset and plot new confusion matrices

In [105]:
x_nonRRDR_nd = x_nonRRDR.to_numpy()
predictions = []

for model_type in all_models_results_original_DT_models.keys():
    for single_model in all_models_results_original_DT_models[model_type]:
        y_proba = single_model.predict_proba(x_nonRRDR_nd)[:,1].astype(int)
        predictions.append(y_proba)

predictions

[array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [106]:
#calculate new confusion matrix for each model
confusion_list = [confusion_matrix(phenotypes_nonRRDR, predictions[i]) for i in range(0,len(predictions))]

In [107]:
cm_mean = np.zeros_like(confusion_list[0],dtype=float)

for result in confusion_list:
    cm_mean += result/len(confusion_list)

cm_mean = np.round(cm_mean)

In [108]:
cm_mean

array([[206.,   0.],
       [ 10.,   0.]])

All mutations are, on average, predicted to be susceptible. 